# BeanSight VN perception pipeline smoke test

This notebook checks the private Hugging Face download, manifest QA, full-frame ROI preprocessing, and one-epoch training path on a Colab GPU. It is not an evaluation run. Do not publish or cite pilot accuracy.

Before running it, add a fine-grained, read-only `HF_TOKEN` under Colab's Secrets panel. Never paste a token into a cell.

In [ ]:
import re

REPOSITORY_URL = "https://github.com/gkienpham-cmd/so101-robot.git"
CODE_REVISION = "REPLACE_WITH_40_CHARACTER_GIT_COMMIT"
DATASET_REPO_ID = "REPLACE_WITH_PRIVATE_HF_DATASET_REPO"
DATASET_REVISION = "REPLACE_WITH_40_CHARACTER_HF_COMMIT"
CAPTURE_METADATA_PATH = "capture_metadata"
BLIND_LABELS_PATH = "blind_labels.csv"
SPLIT_ASSIGNMENTS_PATH = "split_assignments.csv"

assert re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION), "Pin the exact code commit"
assert re.fullmatch(r"[0-9a-f]{40}", DATASET_REVISION), "Pin the exact dataset commit"
assert "REPLACE" not in DATASET_REPO_ID


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add a read-only HF_TOKEN in Colab Secrets"


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="beansight-colab-"))
repo_dir = workspace / "so101-robot"
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", CODE_REVISION], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        f"{repo_dir}[perception]", "huggingface_hub>=0.34",
    ],
    check=True,
)
print(f"Checked out exact code revision {CODE_REVISION}")


In [ ]:
from huggingface_hub import HfApi, snapshot_download

dataset_info = HfApi(token=HF_TOKEN).dataset_info(
    DATASET_REPO_ID, revision=DATASET_REVISION
)
assert dataset_info.sha == DATASET_REVISION, (dataset_info.sha, DATASET_REVISION)
dataset_dir = workspace / "dataset-snapshot"
snapshot_download(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    revision=DATASET_REVISION,
    local_dir=dataset_dir,
    token=HF_TOKEN,
)
print(f"Downloaded exact dataset revision {DATASET_REVISION}")


In [ ]:
qa_dir = workspace / "fresh-qa"
manifest_path = qa_dir / "manifest.csv"
qa_path = qa_dir / "manifest_qa.json"
qa_command = [
    "beansight-build-perception-manifest",
    str(dataset_dir / CAPTURE_METADATA_PATH),
    str(dataset_dir / BLIND_LABELS_PATH),
    "--split-assignments", str(dataset_dir / SPLIT_ASSIGNMENTS_PATH),
    "--config", str(repo_dir / "configs/perception.json"),
    "--output", str(manifest_path),
    "--qa-summary", str(qa_path),
]
subprocess.run(qa_command, check=True)
assert manifest_path.is_file() and qa_path.is_file()
print("Fresh manifest QA passed")


In [ ]:
pilot_output = workspace / "pilot-smoke"
training_command = [
    "beansight-train-perception", str(manifest_path),
    "--config", str(repo_dir / "configs/perception.json"),
    "--dataset-revision", DATASET_REVISION,
    "--epochs", "1",
    "--batch-size", "8",
    "--device", "cuda",
    "--output", str(pilot_output),
]
completed = subprocess.run(training_command, text=True, capture_output=True)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Pilot training path failed")
required_artifacts = {
    "model.ts", "model_state.pt", "metrics.json",
    "operating_threshold.json", "provenance.json",
}
assert required_artifacts <= {path.name for path in pilot_output.iterdir()}
print("Pilot pipeline smoke passed. Keep its metrics private; this is not an evaluation.")


Stop here. Do not upload the pilot checkpoint or quote its accuracy. Scale collection only after the images, blind labels, session boundaries, and held-out lot have passed manual review.